# TF-IDF Doctor Retriever
## Building a Medical Search Engine from Patient Reviews

---

In our Preprocessing notebook, we cleaned **66,465** patient comments into ~52,800 usable rows.

Now we build a **search engine**: given a medical query → return the most relevant doctors.

| Part | Topic |
|------|-------|
| **0** | Setup & Data Loading |
| **1** | TF-IDF Theory (from scratch) |
| **2** | Medical Stopwords Pipeline |
| **3** | Building the Doctor Corpus (medical) |
| **4** | Building the TF-IDF Index |
| **5** | Query Preprocessing |
| **6** | Retrieval — Cosine Similarity Search |
| **7** | Recommendation-Aware Reranking |
| **8** | Holdout Evaluation (Recall@K, MRR) |
| **9** | Qualitative Test Suite |
| **10** | Interactive Search — Play With the Retriever 🎮 |
| **11** | Summary & Next Steps |



The medical variant removes praise ("عالی"), thanks ("ممنون"), clinic ops ("مطب", "نوبت"), and behavioral adjectives ("حوصله", "مهربان") — leaving only **clinical content**.

---
# Part 0: Setup & Data Loading
---

## What is Information Retrieval?

**Information Retrieval (IR)** is the science of finding, within a large collection of unstructured data (usually text), the items that satisfy a user's information need.

```
Google Search:   "kidney stone treatment"   →   Ranked list of web pages
Our System:      "سنگ کلیه"                 →   Ranked list of DOCTORS
```

This is fundamentally different from a database query:

- **Database:** Exact match — `WHERE specialty = 'urology'` returns only urologists, missing doctors whose patients frequently mention kidney stones but whose specialty label is "general surgery".
- **IR:** Semantic relevance — if 50 patients of Dr. X mention "سنگ کلیه" in their reviews, Dr. X is relevant to this query regardless of their listed specialty.

The core idea behind most IR systems: represent both queries and documents as **numerical vectors** in a shared space, then measure the **similarity** between them.

In [1]:
import re, os, math, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

tqdm.pandas(desc="Processing")
plt.rcParams.update({'figure.figsize': (10, 5), 'axes.grid': True, 'grid.alpha': 0.3})
print("All imports OK ✓")

All imports OK ✓


## Loading the Data

We load two files produced by our previous notebooks:

1. **Preprocessed comments** — each row is one patient review, already tokenized and cleaned (output of our Preprocessing notebook).
2. **Doctor metadata** — name, specialty, profile URL (from our EDA notebook).

The key column is `final_preprocessed_text`: a space-separated string of tokens, with standard Persian stopwords already removed and Arabic characters normalized to Persian.

In [2]:
PATH = r"C:\Mohsen Folder\Ai Bootcamp\Ai Bootcamp\data\raw\Training & Education Data\NLP"
# read file (create in 26.NLP - preprocessing)
PATH_COMMENTS = os.path.join(PATH , "comments_for_tfidf_retriever.csv")

comments_df = pd.read_csv(PATH_COMMENTS, keep_default_na=False)

comments_df["doctor_id"] = comments_df["doctor_id"].astype(str)
comments_df["final_preprocessed_text"] = comments_df["final_preprocessed_text"].astype(str)
print(f"Loaded {len(comments_df):,} comments | {comments_df['doctor_id'].nunique():,} doctors")
comments_df.head(3)

Loaded 51,967 comments | 544 doctors


,doctor_id,final_preprocessed_text,rate,label,date
0,100246,زگیل ناخن درمانم,5.0,1,۱۴۰۰/۰۲/۰۶
1,100246,درود تشخیص درست خارش بدن ودرمان,5.0,1,۱۴۰۴/۰۹/۲۸
2,100246,دقت,5.0,1,۱۴۰۴/۰۹/۲۷


In [3]:
comments_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51967 entries, 0 to 51966
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   doctor_id                51967 non-null  str    
 1   final_preprocessed_text  51967 non-null  str    
 2   rate                     51967 non-null  float64
 3   label                    51967 non-null  int64  
 4   date                     51967 non-null  str    
dtypes: float64(1), int64(1), str(3)
memory usage: 2.0 MB


In [4]:
PATH_DOCTIRS = os.path.join(PATH , "doctors_raw_canonical.csv")

doctors_df = pd.read_csv(PATH_DOCTIRS, keep_default_na=False)


doctors_df["doctor_id"] = doctors_df["doctor_id"].astype(str)
name_map = dict(zip(doctors_df["doctor_id"], doctors_df.get("name", ""))) if "name" in doctors_df.columns else {}

specialty_map = dict(zip(doctors_df["doctor_id"], doctors_df.get("specialty", ""))) if "specialty" in doctors_df.columns else {}
print(f"Loaded {len(doctors_df):,} doctor profiles")

Loaded 569 doctor profiles


---
# Part 1: Understanding TF-IDF (from scratch)
---

**TF-IDF** (Term Frequency – Inverse Document Frequency) is one of the most fundamental and widely-used text representation methods in information retrieval. Despite being introduced in the 1970s, it remains a strong baseline that often outperforms more complex methods on small-to-medium corpora.

In this part, we will:
1. Compute TF-IDF **manually** on a tiny 4-sentence corpus
2. Understand **why** each component exists
3. Verify our manual results match `sklearn`

## 1.1 The Core Problem: Representing Text as Numbers

Machine learning algorithms operate on numbers, not words. We need a function:

$$f: \text{"سنگ کلیه عمل درد"} \longrightarrow [0.42, 0.0, 0.31, \ldots] \in \mathbb{R}^{|V|}$$

where $|V|$ is the vocabulary size. Each dimension corresponds to one word (or phrase), and the value encodes how **important** that word is to this document.

**Example:** Query is "سنگ کلیه" (kidney stones).

| Doctor | Comments | Relevance |
|--------|----------|-----------|
| A | سنگ کلیه عمل درد | ✅ High — shares both query terms |
| B | سنگ مثانه درمان | ⚠️ Partial — shares "سنگ" only |
| C | دکتر خوب عالی | ❌ None — no medical overlap |

TF-IDF is a specific instantiation of this function. Let's build it step by step.

## 1.2 Term Frequency (TF)

The simplest idea: **count how often each word appears** in a document.

$$\text{TF}(t, d) = \text{count of term } t \text{ in document } d$$

If "سنگ" appears 5 times in Doctor A's reviews and 1 time in Doctor B's, Doctor A's document has a higher TF for that term — suggesting stronger association with kidney stones.

In [5]:
# Let's build a tiny corpus and compute TF manually
corpus = [
    "سنگ کلیه درد سنگ",    # Doc 0: kidney stone + pain ("سنگ" appears twice)
    "سنگ مثانه درمان",      # Doc 1: bladder stone treatment
    "دکتر خوب عالی",        # Doc 2: irrelevant praise
    "درد کمر دیسک درد",     # Doc 3: back pain + disc ("درد" appears twice)
]
tokenized = [doc.split() for doc in corpus]
print(tokenized)

[['سنگ', 'کلیه', 'درد', 'سنگ'], ['سنگ', 'مثانه', 'درمان'], ['دکتر', 'خوب', 'عالی'], ['درد', 'کمر', 'دیسک', 'درد']]


In [6]:
len(tokenized)

4

In [7]:
# 1. Create an empty set to filter out duplicate words automatically
# 2. Loop through each document (which is a list of words)
# 3. Loop through each word in the current document
# 4. Add the word to the set
# 5. Sort the set alphabetically and convert it to a list

unique_words = set()

for doc in tokenized:
    for w in doc:
        unique_words.add(w)

vocab = sorted(unique_words)
vocab

['خوب', 'درد', 'درمان', 'دکتر', 'دیسک', 'سنگ', 'عالی', 'مثانه', 'کلیه', 'کمر']

In [8]:

# 1. Initialize an empty list to hold the rows of our TF matrix
# 2. Loop through each document in the tokenized corpus
# 3. Count the frequency of each word in the current document using 'Counter'
# 4. Initialize an empty list to represent the current row (document vector)
# 5. Loop through every word in our global vocabulary
# 6. Get the frequency of the vocabulary word in the current document (default to 0 if not found)
# 7. Append this frequency to the current row
# 8. Append the completed row to the main TF matrix
# 9. Create an empty list to store the row labels (indexes) for our DataFrame
# 10. Loop through the total number of documents to generate labels
# 11. Append the label (e.g., 'Doc 0', 'Doc 1') to the indices list
# 12. Create the Pandas DataFrame using the matrix, vocabulary as columns, and generated indices
# 13. Print the vocabulary size and the final DataFrame

tf_matrix = []

for doc in tokenized:
    doc_counts = Counter(doc)
    # print(doc_counts)
    # break
    row = []
    
    for t in vocab:
        count = doc_counts.get(t, 0)
        print(count)
    # break
        row.append(count)
        
    tf_matrix.append(row)

# print(tf_matrix)
# print(len(tf_matrix))
doc_indices = []

for i in range(len(corpus)):
    doc_indices.append(f"Doc {i}")


tf_df = pd.DataFrame(tf_matrix, columns=vocab, index=doc_indices)

print(f"Vocabulary ({len(vocab)} unique terms):\n")
tf_df

0
1
0
0
0
2
0
0
1
0
0
0
1
0
0
1
0
1
0
0
1
0
0
1
0
0
1
0
0
0
0
2
0
0
1
0
0
0
0
1
Vocabulary (10 unique terms):



,خوب,درد,درمان,دکتر,دیسک,سنگ,عالی,مثانه,کلیه,کمر
Doc 0,0,1,0,0,0,2,0,0,1,0
Doc 1,0,0,1,0,0,1,0,1,0,0
Doc 2,1,0,0,1,0,0,1,0,0,0
Doc 3,0,2,0,0,1,0,0,0,0,1


## 1.3 Inverse Document Frequency (IDF)

IDF measures how **informative** a term is across the entire corpus. The intuition: a word that appears in *every* document carries no information for distinguishing between them.

$$\text{IDF}(t) = \log\left(\frac{1 + N}{1 + \text{df}(t)}\right) + 1$$

Where:
- $N$ = total number of documents in the corpus
- $\text{df}(t)$ = **document frequency** — number of documents containing term $t$

**Key insight:** The logarithm ensures that IDF grows slowly. A term appearing in 1 out of 1000 documents doesn't get 1000× the weight of a term appearing in every document — the log compresses this to a manageable scale.

The `+1` additions serve as **smoothing**: the one inside the fraction prevents division by zero; the one outside prevents IDF from ever being zero (every term gets at least some weight).

> 📝 This is sklearn's default "smooth IDF" formula. Other variants exist (e.g., standard IDF without the +1 additions), but this one is most robust in practice.

In [9]:
corpus

['سنگ کلیه درد سنگ', 'سنگ مثانه درمان', 'دکتر خوب عالی', 'درد کمر دیسک درد']

In [10]:
# 1. Get the total number of documents in our corpus (N)
# 2. --- Calculate Document Frequency (DF) ---
# 3. Create an empty dictionary to store the document frequencies
# 4. Loop through each word in the vocabulary
# 5. Initialize a counter for the current word
# 6. Loop through each document in the tokenized corpus
# 7. If the word exists in the current document, increment the counter
# 8. Store the final count in the doc_freq dictionary for that word
# 9. --- Calculate Inverse Document Frequency (IDF) ---
# 10. Create an empty dictionary to store the IDF values
# 11. Loop through each word in the vocabulary again
# 12. Apply the Scikit-Learn smooth IDF formula: log((1 + N) / (1 + DF)) + 1
# 13. Round the result to 4 decimal places and store it in the idf_values dictionary
# 14. --- Display the Results ---
# 15. Create and print a DataFrame to display DF and IDF side-by-side
# 16. Find and print the most and least informative words based on their max/min IDF scores

import numpy as np
import pandas as pd

N = len(corpus) # ['سنگ کلیه درد سنگ', 'سنگ مثانه درمان', 'دکتر خوب عالی', 'درد کمر دیسک درد']

doc_freq = {}
for t in vocab:
    count = 0
    for doc in tokenized:
        if t in doc:
            count += 1
    doc_freq[t] = count
# print(doc_freq)

idf_values = {}
for t in vocab:
    # Applying: log((1 + N) / (1 + df(t))) + 1
    math_formula = np.log((1 + N) / (1 + doc_freq[t])) + 1
    idf_values[t] = round(math_formula, 4)

# print(idf_values)
print(pd.DataFrame({"df (in how many docs?)": doc_freq, "IDF": idf_values}).to_string())

# Find the key (word) with the maximum and minimum IDF values
most_informative = max(idf_values, key=idf_values.get)
least_informative = min(idf_values, key=idf_values.get)

print(f"\n→ Most informative:  '{most_informative}' — appears in fewest docs")
print(f"→ Least informative: '{least_informative}' — appears in most docs")

       df (in how many docs?)     IDF
خوب                         1  1.9163
درد                         2  1.5108
درمان                       1  1.9163
دکتر                        1  1.9163
دیسک                        1  1.9163
سنگ                         2  1.5108
عالی                        1  1.9163
مثانه                       1  1.9163
کلیه                        1  1.9163
کمر                         1  1.9163

→ Most informative:  'خوب' — appears in fewest docs
→ Least informative: 'درد' — appears in most docs


In [11]:
N = len(corpus)
doc_freq = {t: sum(1 for doc in tokenized if t in doc) for t in vocab}
idf_values = {t: round(np.log((1 + N) / (1 + doc_freq[t])) + 1, 4) for t in vocab}

print(pd.DataFrame({"df (in how many docs?)": doc_freq, "IDF": idf_values}).to_string())

print(f"\n→ Most informative:  '{max(idf_values, key=idf_values.get)}' — appears in fewest docs")
print(f"→ Least informative: '{min(idf_values, key=idf_values.get)}' — appears in most docs")

       df (in how many docs?)     IDF
خوب                         1  1.9163
درد                         2  1.5108
درمان                       1  1.9163
دکتر                        1  1.9163
دیسک                        1  1.9163
سنگ                         2  1.5108
عالی                        1  1.9163
مثانه                       1  1.9163
کلیه                        1  1.9163
کمر                         1  1.9163

→ Most informative:  'خوب' — appears in fewest docs
→ Least informative: 'درد' — appears in most docs


## 1.4 TF × IDF: The Complete Picture

Now we combine both signals by multiplying them:

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

This produces a **weighted** term-document matrix where:
- **Rare, relevant terms** (high TF in this doc, high IDF globally) → **high** score
- **Common, uninformative terms** (high TF, but low IDF) → **moderate** score
- **Absent terms** → **zero**

In [14]:
a = ["a", "b", "c","a", "b", "c","a", "b","a"]
Counter(a)

Counter({'a': 4, 'b': 3, 'c': 2})

In [12]:
# 1. Initialize an empty list to store the final TF-IDF matrix
# 2. Loop through each document in the tokenized corpus
# 3. Count the occurrences of each word in the current document (This is our TF)
# 4. Create an empty list to represent the TF-IDF vector for the current document (row)
# 5. Loop through every word in our global vocabulary
# 6. Get the Term Frequency (TF) for the current word in this document (default to 0 if missing)
# 7. Get the Inverse Document Frequency (IDF) for the current word from our previously calculated dictionary
# 8. Multiply TF by IDF to get the TF-IDF score, and round it to 4 decimal places
# 9. Append this score to the current document's vector (row)
# 10. Append the completed document vector to the main TF-IDF matrix
# 11. Create a list of labels for the DataFrame index (e.g., 'Doc 0', 'Doc 1')
# 12. Create a Pandas DataFrame using the TF-IDF matrix, with vocabulary as columns
# 13. Print the title and display the DataFrame

tfidf_matrix = []

for doc in tokenized:
    doc_tf_counts = Counter(doc)
    # print(doc_tf_counts)
    # break
    doc_tfidf_vector = []
    
    for t in vocab:
        tf = doc_tf_counts.get(t, 0)
        print(tf)
        # break
    # break
        idf = idf_values[t]
        
        # The core magic: Local Frequency * Global Importance
        tfidf_score = tf * idf
        
        rounded_score = round(tfidf_score, 4)
        doc_tfidf_vector.append(rounded_score)
    print(doc_tfidf_vector)
    # break        
    tfidf_matrix.append(doc_tfidf_vector)


doc_indices = []
for i in range(len(corpus)):
    doc_indices.append(f"Doc {i}")

tfidf_df = pd.DataFrame(tfidf_matrix, columns=vocab, index=doc_indices)

print("TF-IDF Matrix (before normalization):")
tfidf_df

0
1
0
0
0
2
0
0
1
0
[np.float64(0.0), np.float64(1.5108), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(3.0216), np.float64(0.0), np.float64(0.0), np.float64(1.9163), np.float64(0.0)]
0
0
1
0
0
1
0
1
0
0
[np.float64(0.0), np.float64(0.0), np.float64(1.9163), np.float64(0.0), np.float64(0.0), np.float64(1.5108), np.float64(0.0), np.float64(1.9163), np.float64(0.0), np.float64(0.0)]
1
0
0
1
0
0
1
0
0
0
[np.float64(1.9163), np.float64(0.0), np.float64(0.0), np.float64(1.9163), np.float64(0.0), np.float64(0.0), np.float64(1.9163), np.float64(0.0), np.float64(0.0), np.float64(0.0)]
0
2
0
0
1
0
0
0
0
1
[np.float64(0.0), np.float64(3.0216), np.float64(0.0), np.float64(0.0), np.float64(1.9163), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.9163)]
TF-IDF Matrix (before normalization):


,خوب,درد,درمان,دکتر,دیسک,سنگ,عالی,مثانه,کلیه,کمر
Doc 0,0.0000,1.5108,0.0000,0.0000,0.0000,3.0216,0.0000,0.0000,1.9163,0.0000
Doc 1,0.0000,0.0000,1.9163,0.0000,0.0000,1.5108,0.0000,1.9163,0.0000,0.0000
Doc 2,1.9163,0.0000,0.0000,1.9163,0.0000,0.0000,1.9163,0.0000,0.0000,0.0000
Doc 3,0.0000,3.0216,0.0000,0.0000,1.9163,0.0000,0.0000,0.0000,0.0000,1.9163
